In [1]:
import pyEDM
import pandas as pd
import numpy as np
import random
from collections import Counter

p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [2]:
# top 20 features from paper + target variable
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    "Solidity", "texture_uniformity", "texture_smoothness",
    "texture_average_gray_level", "texture_entropy",
    "texture_average_contrast", "H90", "H180", "Hflip",
    "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")
df = df.asfreq("D")

df_filled = df.copy()

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    df_norm[col] = (df_filled[col] - mu) / sigma

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
df_mv = df_mv[["t"] + features]

target = "Lpoly_expected_ml"
predictors = [col for col in features if col != target]

df_mv.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,texture_average_gray_level,texture_entropy,texture_average_contrast,H90,H180,Hflip,Extent,EquivDiameter,ConvexArea,ConvexPerimeter
0,1,0.285287,1.059338,0.967595,0.956625,1.139942,1.226418,0.842266,-1.228078,-0.191169,...,-1.183856,0.480870,-1.005885,-0.901014,0.861091,1.128490,0.102206,1.092577,1.111408,1.118301
1,2,0.065656,1.016480,0.917837,0.908034,1.109145,1.007938,0.563761,-1.247322,0.526462,...,-1.141946,0.640061,-0.878977,-1.001016,0.340277,0.680457,0.656972,1.062576,0.999125,1.020599
2,3,-0.128818,0.671753,0.512658,0.637894,0.852846,0.719154,0.645106,-1.070290,0.486697,...,-1.073251,0.996070,-0.759717,-0.976040,0.078281,0.314316,0.520649,0.807369,0.659696,0.771253
3,4,-0.018094,0.379651,0.165389,0.433051,0.632838,0.501162,0.894757,-0.832234,0.370645,...,-1.029278,1.106370,-0.779254,-0.852571,0.097279,0.420381,0.192816,0.595165,0.375226,0.570793
4,5,0.891676,0.756008,0.593485,0.724495,0.928235,0.917121,0.418304,-1.071257,0.091247,...,-1.184920,0.232139,-0.948879,-0.995896,0.312944,0.654387,0.256093,0.879835,0.773202,0.863842


In [3]:
env = pd.read_csv("../data/environment_all.csv")

# could be relevant but too many missing values for ema imputation to be effective
env = env.drop(columns=[
    'fluorescent_dissolved_organic_matter_eco',
    'sea_water_turbidity_eco',
    'waterlevel_predicted_m',
    'mass_concentration_of_oxygen_in_sea_water_seaphox',
    'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox',
    'fractional_saturation_of_oxygen_in_sea_water_seaphox',
    'sea_water_ph_reported_on_total_scale_seaphox_external'
])

env["date"] = pd.to_datetime(env["date"].astype(str), format="%Y%m%d")
env = env.sort_values("date").set_index("date")
env = env.asfreq("D")

env_features = env.columns.tolist()
env_filled = env.copy()

for col in env_features:
    ema = env[col].ewm(span=30, adjust=False).mean()
    env_filled[col] = env[col].fillna(ema)

df_norm = env_filled.copy()

for col in env_features:
    mu = env_filled[col].mean()
    sigma = env_filled[col].std()
    df_norm[col] = (env_filled[col] - mu) / sigma

df_env = df_norm.copy()
df_env = df_env.reset_index()
df_env["t"] = np.arange(1, len(df_env) + 1)
df_env = df_env[["t"] + env_features]

df_all = pd.merge(df_mv, df_env, on="t", how="inner")

df_env = df_env.merge(df_mv[["t", target]], on="t", how="inner")
df_env.head()

,t,mass_concentration_of_chlorophyll_in_sea_water_ctd,sea_water_electrical_conductivity_ctd,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m,Lpoly_expected_ml
0,1,0.386650,1.505381,0.361222,-0.887083,0.781443,1.526531,-0.299634,0.288179,-0.014762,1.008903,-1.294158,1.006837,0.285287
1,2,0.718488,1.473634,0.375660,-0.833821,0.519131,1.488148,-0.107828,0.303818,-0.095281,1.224096,-0.784883,0.654760,0.065656
2,3,1.065658,0.860840,0.326331,-0.349615,0.544113,0.835239,-0.115066,0.714121,-0.069400,1.365032,-0.344831,0.304790,-0.128818
3,4,1.402899,0.352516,0.254143,0.008697,0.331765,0.303019,-0.799053,-0.682379,-0.785444,1.101345,-0.532718,0.348009,-0.018094
4,5,1.678048,0.440744,0.238502,-0.079670,0.181872,0.407880,-0.665150,0.782198,-0.788319,0.808864,-0.370789,0.219406,0.891676


In [4]:
df_all.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m
0,1,0.285287,1.059338,0.967595,0.956625,1.139942,1.226418,0.842266,-1.228078,-0.191169,...,0.361222,-0.887083,0.781443,1.526531,-0.299634,0.288179,-0.014762,1.008903,-1.294158,1.006837
1,2,0.065656,1.016480,0.917837,0.908034,1.109145,1.007938,0.563761,-1.247322,0.526462,...,0.375660,-0.833821,0.519131,1.488148,-0.107828,0.303818,-0.095281,1.224096,-0.784883,0.654760
2,3,-0.128818,0.671753,0.512658,0.637894,0.852846,0.719154,0.645106,-1.070290,0.486697,...,0.326331,-0.349615,0.544113,0.835239,-0.115066,0.714121,-0.069400,1.365032,-0.344831,0.304790
3,4,-0.018094,0.379651,0.165389,0.433051,0.632838,0.501162,0.894757,-0.832234,0.370645,...,0.254143,0.008697,0.331765,0.303019,-0.799053,-0.682379,-0.785444,1.101345,-0.532718,0.348009
4,5,0.891676,0.756008,0.593485,0.724495,0.928235,0.917121,0.418304,-1.071257,0.091247,...,0.238502,-0.079670,0.181872,0.407880,-0.665150,0.782198,-0.788319,0.808864,-0.370789,0.219406


https://sugiharalab.github.io/EDM_Documentation/parameters/

In [5]:
def one_simplex(df, target, features, E=4, Tp=1):
    # Randomly select 3 features (+ the target) for the simplex projection
    chosen_features = random.sample(features, 3)
    columns = [target] + chosen_features
    columns_str = " ".join(columns) # has to be 'space separated' idk ????

    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        columns=columns_str,
        target=target,
        E=E,
        tau=1,
        Tp=Tp,
        lib=f"1 {N}",
        pred=f"1 {N}"
    )

    obs = res["Observations"].to_numpy()
    pred = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]

    if len(obs) < 10 or np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan, chosen_features

    rho = np.corrcoef(obs, pred)[0, 1]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mae = np.mean(np.abs(obs - pred))
    return rho, rmse, mae, chosen_features

def multiview_big(df, target, features, Tp, n_trials=500):
    results = []

    for i in range(n_trials):
        rho, rmse, mae, chosen = one_simplex(df, target, features, E=4, Tp=Tp)

        results.append({
            "rho": rho,
            "rmse": rmse,
            "mae": mae,
            "features": chosen
        })

    return pd.DataFrame(results)

def multiview_yes(df_mv, target, predictors):
    # wrapper main function
    x = df_mv[target].to_numpy()

    summary_rows = []
    feature_importance_by_tp = {}
    
    for Tp in range(1, 32):
        mv = multiview_big(df_mv, target, predictors, Tp, n_trials=500)

        acf = abs(pd.Series(x).autocorr(lag=Tp))
        mv["rho_eff"] = mv["rho"] - acf

        # summary stats
        summary_rows.append({
            "Tp": Tp,
            "rho_eff_mean": mv["rho_eff"].mean(),
            "rmse_mean": mv["rmse"].mean(),
            "mae_mean": mv["mae"].mean(),
            "n_models": len(mv)
        })

        # feature importance (top 5%)
        top = mv.nlargest(int(0.05 * len(mv)), "rho_eff")

        counts = Counter()
        for feats in top["features"]:
            for f in feats:
                counts[f] += 1

        importance = pd.Series(counts) / len(top)
        feature_importance_by_tp[Tp] = importance.sort_values(ascending=False)
    
    return summary_rows, feature_importance_by_tp

In [6]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_mv, target, predictors)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.189656   0.439709  0.083556       500
1    2      0.442174   0.402618  0.078741       500
2    3      0.584710   0.378149  0.080612       500
3    4      0.264098   0.814828  0.181922       500
4    5      0.125081   0.945992  0.217354       500
5    6      0.171235   0.963141  0.225739       500
6    7      0.136112   0.983660  0.236607       500
7    8      0.083668   0.997850  0.239958       500
8    9      0.100127   0.998733  0.241805       500
9   10      0.076194   0.998020  0.243516       500
10  11      0.077460   1.002566  0.246495       500
11  12      0.062669   1.023266  0.265049       500
12  13      0.018650   1.044766  0.279242       500
13  14      0.058662   1.051932  0.287308       500
14  15      0.096818   1.047245  0.290596       500
15  16      0.133548   1.030815  0.294512       500
16  17      0.170171   1.007973  0.293799       500
17  18      0.262744   0.974165  0.292853       500
18  19      

In [7]:
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)

importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
print(importance_all)

     Tp                     Feature  Proportion
0     1                 Orientation        0.56
1     1  texture_average_gray_level        0.36
2     1                        Area        0.36
3     1                  ConvexArea        0.36
4     1                      Extent        0.24
..   ..                         ...         ...
469  31  texture_average_gray_level        0.12
470  31                   Biovolume        0.12
471  31                         H90        0.08
472  31          texture_smoothness        0.08
473  31                   Perimeter        0.04

[474 rows x 3 columns]


In [8]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_env, target, env_features)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.179783   0.457985  0.118642       500
1    2      0.434915   0.421410  0.114056       500
2    3      0.575303   0.403490  0.116839       500
3    4      0.269524   0.809906  0.203614       500
4    5      0.133094   0.930502  0.231525       500
5    6      0.193659   0.940743  0.239015       500
6    7      0.194966   0.954506  0.248537       500
7    8      0.111253   0.994328  0.261211       500
8    9      0.142561   0.976518  0.256089       500
9   10      0.116023   0.984743  0.267135       500
10  11      0.143751   0.989065  0.276241       500
11  12      0.206407   0.983513  0.286396       500
12  13      0.217397   0.978963  0.293400       500
13  14      0.271726   0.971744  0.289428       500
14  15      0.277634   0.981469  0.291807       500
15  16      0.257713   0.983499  0.296795       500
16  17      0.228238   0.981302  0.297972       500
17  18      0.217540   0.992216  0.294025       500
18  19      

In [9]:
importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
print(importance_all)

     Tp                                            Feature  Proportion
0     1                          sea_water_temperature_ctd        0.76
1     1                              sea_water_sigma_t_ctd        0.72
2     1              sea_water_electrical_conductivity_ctd        0.60
3     1                   sea_water_practical_salinity_ctd        0.28
4     1                                         air_temp_c        0.24
..   ..                                                ...         ...
233  31                              sea_water_sigma_t_ctd        0.60
234  31                   sea_water_practical_salinity_ctd        0.60
235  31                          sea_water_temperature_ctd        0.44
236  31  mass_concentration_of_chlorophyll_in_sea_water...        0.36
237  31              sea_water_electrical_conductivity_ctd        0.28

[238 rows x 3 columns]


In [10]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_all, target, predictors+env_features)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.186222   0.446905  0.095195       500
1    2      0.439606   0.409230  0.089485       500
2    3      0.580747   0.388164  0.092033       500
3    4      0.262693   0.816064  0.187573       500
4    5      0.123097   0.945288  0.222194       500
5    6      0.180408   0.955104  0.228022       500
6    7      0.172178   0.966622  0.235366       500
7    8      0.103161   0.992116  0.241269       500
8    9      0.127162   0.983789  0.239816       500
9   10      0.100002   0.986049  0.242375       500
10  11      0.108996   0.993209  0.248919       500
11  12      0.120356   1.006486  0.267359       500
12  13      0.110561   1.010550  0.276121       500
13  14      0.158562   1.012639  0.278925       500
14  15      0.195110   1.013895  0.283835       500
15  16      0.204357   1.010953  0.290876       500
16  17      0.223572   0.992100  0.290467       500
17  18      0.220501   1.001697  0.294466       500
18  19      